## 1. Install & Import Dependencies

In [1]:
# ── Install any missing libraries ──────────────────────────────────────────
import subprocess, sys

PACKAGES = [
    "torch",
    "torchvision",
    "transformers",
    "datasets",
    "scikit-learn",
    "Pillow",
    "matplotlib",
    "numpy",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])
print("✅ All packages ready.")

✅ All packages ready.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import argparse
import copy
import csv
import json
import random
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from torch import nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from transformers import AutoProcessor, CLIPModel

matplotlib.use("Agg")  # non-interactive backend for saving figures
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.11.0 | CUDA available: False


---
## 2. Configuration

Edit the `ExperimentConfig` fields below to customise the run.

| Parameter | Default | Description |
|---|---|---|
| `model_id` | `openai/clip-vit-base-patch32` | HuggingFace CLIP checkpoint |
| `top_k` | `10` | Number of Food101 classes to select |
| `shots_per_class` | `8` | Few-shot training examples per class |
| `val_per_class` | `20` | Dev-set examples per class (used for early stopping) |
| `epochs` | `20` | Linear probe training epochs |
| `learning_rate` | `1e-3` | AdamW learning rate |

In [3]:
DEFAULT_PROMPT_TEMPLATES = (
    "a photo of {}.",
    "a close-up photo of {}.",
    "a photo of a plate of {}.",
    "a food photo of {}.",
    "a restaurant photo of {}.",
)


@dataclass
class ExperimentConfig:
    model_id: str = "openai/clip-vit-base-patch32"
    dataset_name: str = "ethz/food101"
    top_k: int = 10
    class_names: Optional[List[str]] = None   # None → pick alphabetically
    shots_per_class: int = 8
    val_per_class: int = 20
    batch_size: int = 16
    num_workers: int = 0
    epochs: int = 20
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    seed: int = 42
    device: Optional[str] = None             # None → auto-detect
    output_root: Path = Path("outputs/food101_experiments")
    prompt_templates: Tuple[str, ...] = field(
        default_factory=lambda: tuple(DEFAULT_PROMPT_TEMPLATES)
    )

    def to_serializable_dict(self) -> Dict[str, Any]:
        payload = asdict(self)
        payload["output_root"] = str(self.output_root)
        payload["prompt_templates"] = list(self.prompt_templates)
        return payload


# ── Instantiate config (edit here) ─────────────────────────────────────────
CFG = ExperimentConfig(
    model_id="openai/clip-vit-base-patch32",
    top_k=10,
    class_names=[
        "apple_pie",
        "bibimbap",
        "chicken_wings",
        "donuts",
        "eggs_benedict",
        "french_fries",
        "grilled_cheese_sandwich",
        "hamburger",
        "ice_cream",
        "pizza",
    ],
    shots_per_class=128,
    val_per_class=20,
    batch_size=16,
    epochs=20,
    learning_rate=1e-3,
    weight_decay=1e-4,
    output_root=Path("output/clip-vit-base-patch32"),
)


print(json.dumps(CFG.to_serializable_dict(), indent=2))

{
  "model_id": "openai/clip-vit-base-patch32",
  "dataset_name": "ethz/food101",
  "top_k": 10,
  "class_names": [
    "apple_pie",
    "bibimbap",
    "chicken_wings",
    "donuts",
    "eggs_benedict",
    "french_fries",
    "grilled_cheese_sandwich",
    "hamburger",
    "ice_cream",
    "pizza"
  ],
  "shots_per_class": 128,
  "val_per_class": 20,
  "batch_size": 16,
  "num_workers": 0,
  "epochs": 20,
  "learning_rate": 0.001,
  "weight_decay": 0.0001,
  "seed": 42,
  "device": null,
  "output_root": "output/clip-vit-base-patch32",
  "prompt_templates": [
    "a photo of {}.",
    "a close-up photo of {}.",
    "a photo of a plate of {}.",
    "a food photo of {}.",
    "a restaurant photo of {}."
  ]
}


---
## 3. Utility Functions

In [4]:
# ── Name helpers ───────────────────────────────────────────────────────────
def canonicalize_class_name(name: str) -> str:
    return name.strip().lower().replace(" ", "_")


def humanize_class_name(name: str) -> str:
    return name.replace("_", " ")


# ── Device resolution ──────────────────────────────────────────────────────
def resolve_device(requested_device: Optional[str] = None) -> torch.device:
    if requested_device:
        return torch.device(requested_device)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


# ── Reproducibility ────────────────────────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ── Config validation ──────────────────────────────────────────────────────
def validate_config(config: ExperimentConfig) -> None:
    checks = [
        (config.top_k > 0,           "top_k must be positive."),
        (config.shots_per_class > 0, "shots_per_class must be positive."),
        (config.val_per_class >= 0,  "val_per_class must be >= 0."),
        (config.batch_size > 0,      "batch_size must be positive."),
        (config.epochs > 0,          "epochs must be positive."),
        (config.learning_rate > 0,   "learning_rate must be positive."),
        (config.weight_decay >= 0,   "weight_decay must be >= 0."),
        (bool(config.prompt_templates), "At least one prompt template is required."),
    ]
    for condition, message in checks:
        if not condition:
            raise ValueError(message)
    if any("{}" not in t for t in config.prompt_templates):
        raise ValueError("Every prompt template must include a {} placeholder.")
    print("Config validated.")


# ── Output directory ───────────────────────────────────────────────────────
def create_output_dir(output_root: Path) -> Path:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_dir = output_root / timestamp
    output_dir.mkdir(parents=True, exist_ok=False)
    return output_dir


# ── Persistence helpers ────────────────────────────────────────────────────
def save_json(path: Path, payload: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)


def save_rows_csv(
    path: Path, rows: List[Dict[str, Any]], fieldnames: List[str]
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def save_confusion_matrix_csv(
    path: Path, matrix: np.ndarray, class_names: Sequence[str]
) -> None:
    readable = [humanize_class_name(n) for n in class_names]
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["true_label"] + readable)
        for name, row in zip(readable, matrix.tolist()):
            writer.writerow([name] + row)


print("Utility functions defined.")

Utility functions defined.


---
## 4. Visualisation Utilities

In [5]:
def save_confusion_matrix_plot(
    path: Path,
    matrix: np.ndarray,
    class_names: Sequence[str],
    title: str,
    show: bool = True,
) -> None:
    readable = [humanize_class_name(n) for n in class_names]
    fig_w = max(8, len(class_names) * 1.1)
    fig_h = max(6, len(class_names) * 0.9)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    img = ax.imshow(matrix, cmap="Blues")
    fig.colorbar(img, ax=ax)
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    ax.set_xticks(np.arange(len(readable)))
    ax.set_yticks(np.arange(len(readable)))
    ax.set_xticklabels(readable, rotation=45, ha="right")
    ax.set_yticklabels(readable)
    threshold = matrix.max() / 2 if matrix.size else 0.0
    for r in range(matrix.shape[0]):
        for c in range(matrix.shape[1]):
            color = "white" if matrix[r, c] > threshold else "black"
            ax.text(c, r, int(matrix[r, c]), ha="center", va="center",
                    color=color, fontsize=8)
    fig.tight_layout()
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    if show:
        plt.show()
    plt.close(fig)


def plot_training_history(history_rows: List[Dict[str, Any]], output_dir: Path) -> None:
    """
    Plot loss and accuracy curves from the linear probe training history.
    Saves the figure and displays it inline.
    """
    epochs   = [r["epoch"] for r in history_rows]
    train_loss = [r["train_loss"] for r in history_rows]
    has_dev  = "dev_loss" in history_rows[0]

    n_plots = 3 if has_dev else 1
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]

    # Train loss
    axes[0].plot(epochs, train_loss, marker="o", label="Train loss")
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    if has_dev:
        dev_loss = [r["dev_loss"] for r in history_rows]
        dev_acc  = [r["dev_accuracy"] for r in history_rows]
        dev_f1   = [r.get("dev_macro_f1", None) for r in history_rows]

        axes[1].plot(epochs, dev_loss, marker="o", color="orange", label="Dev loss")
        axes[1].set_title("Dev Loss")
        axes[1].set_xlabel("Epoch")
        axes[1].set_ylabel("Loss")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        axes[2].plot(epochs, dev_acc, marker="o", color="green", label="Dev accuracy")
        if any(v is not None for v in dev_f1):
            axes[2].plot(epochs, dev_f1, marker="s", color="purple",
                         linestyle="--", label="Dev macro-F1")
        axes[2].set_title("Dev Metrics")
        axes[2].set_xlabel("Epoch")
        axes[2].set_ylabel("Score")
        axes[2].set_ylim(0, 1)
        axes[2].legend()
        axes[2].grid(alpha=0.3)

    fig.tight_layout()
    out_path = output_dir / "training_curves.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  → Training curves saved to {out_path}")


def plot_comparison_bar(
    zero_shot_metrics: Dict[str, float],
    few_shot_metrics: Dict[str, float],
    output_dir: Path,
) -> None:
    """
    Side-by-side bar chart comparing zero-shot vs few-shot on key metrics.
    """
    keys = ["accuracy", "balanced_accuracy", "macro_f1", "weighted_f1"]
    labels = ["Accuracy", "Bal. Accuracy", "Macro F1", "Weighted F1"]
    x = np.arange(len(keys))
    width = 0.35

    zs_vals  = [zero_shot_metrics.get(k, 0) for k in keys]
    fs_vals  = [few_shot_metrics.get(k, 0)  for k in keys]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars_zs = ax.bar(x - width / 2, zs_vals, width, label="Zero-shot",
                     color="steelblue", alpha=0.85)
    bars_fs = ax.bar(x + width / 2, fs_vals, width, label="Few-shot (linear probe)",
                     color="darkorange", alpha=0.85)

    for bar in bars_zs:
        ax.annotate(f"{bar.get_height():.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)
    for bar in bars_fs:
        ax.annotate(f"{bar.get_height():.3f}",
                    xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.set_title("Zero-Shot vs Few-Shot: Metric Comparison", fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    out_path = output_dir / "comparison_bar.png"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"  → Comparison bar chart saved to {out_path}")


print("Visualisation utilities defined.")

Visualisation utilities defined.


---
## 5. Dataset: Class & Collate

In [6]:
class Food101SubsetDataset(Dataset):
    """
    Wraps a HuggingFace Food101 split, remapping original label IDs
    to a contiguous 0-based index over the selected subset of classes.
    """

    def __init__(
        self,
        dataset_split: Any,
        original_to_new: Dict[int, int],
        class_names: Sequence[str],
        split_name: str,
    ) -> None:
        self.dataset_split  = dataset_split
        self.original_to_new = original_to_new
        self.class_names    = list(class_names)
        self.split_name     = split_name

    def __len__(self) -> int:
        return len(self.dataset_split)

    def __getitem__(self, index: int) -> Dict[str, Any]:
        example = self.dataset_split[index]
        image   = example["image"]
        if not isinstance(image, Image.Image):
            raise TypeError(f"Expected a PIL image, got {type(image)!r}")
        label_id = self.original_to_new[example["label"]]
        return {
            "index":      index,
            "image":      image.convert("RGB"),
            "label":      label_id,
            "label_name": self.class_names[label_id],
            "split_name": self.split_name,
        }


def image_collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, Any]:
    return {
        "indices":     torch.tensor([item["index"]  for item in batch], dtype=torch.long),
        "images":      [item["image"]      for item in batch],
        "labels":      torch.tensor([item["label"]  for item in batch], dtype=torch.long),
        "label_names": [item["label_name"] for item in batch],
        "split_names": [item["split_name"] for item in batch],
    }


print("Dataset class ready.")

Dataset class ready.


---
## 6. Data Loading & Splitting

In [7]:
def select_classes(
    available_class_names: Sequence[str], config: ExperimentConfig
) -> List[str]:
    available_set = {canonicalize_class_name(n) for n in available_class_names}
    if config.class_names:
        selected = [canonicalize_class_name(n) for n in config.class_names]
    else:
        selected = sorted(available_set)[: config.top_k]
    if len(selected) != len(set(selected)):
        raise ValueError("Class names must be unique.")
    if len(selected) == 0:
        raise ValueError("At least one class must be selected.")
    missing = sorted(set(selected) - available_set)
    if missing:
        raise ValueError(f"Unknown classes: {missing}")
    return selected


def filter_dataset_split(
    dataset_split: Any, selected_original_ids: Sequence[int]
) -> Any:
    selected_set = set(selected_original_ids)
    labels = dataset_split["label"]
    keep = [i for i, lbl in enumerate(labels) if lbl in selected_set]
    return dataset_split.select(keep)


def build_few_shot_indices(
    labels: Sequence[int],
    selected_original_ids: Sequence[int],
    shots_per_class: int,
    val_per_class: int,
    seed: int,
) -> Tuple[List[int], List[int]]:
    grouped: Dict[int, List[int]] = defaultdict(list)
    for idx, lbl in enumerate(labels):
        grouped[lbl].append(idx)

    rng = random.Random(seed)
    train_idx, val_idx = [], []
    required = shots_per_class + val_per_class

    for orig_lbl in selected_original_ids:
        idxs = list(grouped[orig_lbl])
        rng.shuffle(idxs)
        if len(idxs) < required:
            raise ValueError(
                f"Not enough samples for label {orig_lbl}: "
                f"need {required}, found {len(idxs)}."
            )
        train_idx.extend(idxs[:shots_per_class])
        val_idx.extend(idxs[shots_per_class:required])

    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    return train_idx, val_idx


def load_food101_datasets(
    config: ExperimentConfig,
) -> Tuple[Dict[str, Dataset], List[str], Dict[str, Any]]:
    dataset = load_dataset(config.dataset_name)
    available_splits = sorted(dataset.keys())
    if "train" not in dataset or "validation" not in dataset:
        raise ValueError(
            f"Expected 'train' and 'validation' splits, found: {available_splits}"
        )

    all_class_names      = dataset["train"].features["label"].names
    selected_class_names = select_classes(all_class_names, config)

    name_to_orig = {
        canonicalize_class_name(n): i for i, n in enumerate(all_class_names)
    }
    selected_orig_ids = [name_to_orig[n] for n in selected_class_names]
    original_to_new   = {oid: nid for nid, oid in enumerate(selected_orig_ids)}

    filtered_train = filter_dataset_split(dataset["train"],      selected_orig_ids)
    filtered_val   = filter_dataset_split(dataset["validation"], selected_orig_ids)

    fs_train_idx, fs_dev_idx = build_few_shot_indices(
        labels=filtered_train["label"],
        selected_original_ids=selected_orig_ids,
        shots_per_class=config.shots_per_class,
        val_per_class=config.val_per_class,
        seed=config.seed,
    )

    train_ds = Food101SubsetDataset(
        filtered_train.select(fs_train_idx), original_to_new,
        selected_class_names, "few_shot_train"
    )
    dev_ds = Food101SubsetDataset(
        filtered_train.select(fs_dev_idx), original_to_new,
        selected_class_names, "few_shot_dev"
    )
    eval_ds = Food101SubsetDataset(
        filtered_val, original_to_new, selected_class_names, "validation"
    )

    summary = {
        "dataset_name":              config.dataset_name,
        "available_splits":          available_splits,
        "selected_class_names":      selected_class_names,
        "selected_class_names_readable": [
            humanize_class_name(n) for n in selected_class_names
        ],
        "shots_per_class":           config.shots_per_class,
        "dev_per_class":             config.val_per_class,
        "few_shot_train_size":       len(train_ds),
        "few_shot_dev_size":         len(dev_ds),
        "held_out_validation_size":  len(eval_ds),
    }

    return (
        {"few_shot_train": train_ds, "few_shot_dev": dev_ds, "validation": eval_ds},
        selected_class_names,
        summary,
    )


print(" Data loading functions defined.")

 Data loading functions defined.


In [8]:
# ── Actually load the data ─────────────────────────────────────────────────
validate_config(CFG)
set_seed(CFG.seed)
DEVICE     = resolve_device(CFG.device)
OUTPUT_DIR = create_output_dir(CFG.output_root)

print(f"Device   : {DEVICE}")
print(f"Output   : {OUTPUT_DIR}")
print()
print(f"Loading dataset '{CFG.dataset_name}'...")
datasets_by_split, CLASS_NAMES, dataset_summary = load_food101_datasets(CFG)

dataset_summary["device"] = str(DEVICE)
save_json(OUTPUT_DIR / "dataset_summary.json", dataset_summary)
save_json(OUTPUT_DIR / "config.json", CFG.to_serializable_dict())

print("\n Dataset split sizes:")
for split_name, ds in datasets_by_split.items():
    print(f"   {split_name:<20} → {len(ds):>5} examples")
print(f"\n  Selected classes ({len(CLASS_NAMES)}):")
for i, cn in enumerate(CLASS_NAMES):
    print(f"   [{i:>2}] {humanize_class_name(cn)}")

Config validated.
Device   : mps
Output   : output/clip-vit-base-patch32/20260326_170105

Loading dataset 'ethz/food101'...

 Dataset split sizes:
   few_shot_train       →  1280 examples
   few_shot_dev         →   200 examples
   validation           →  2500 examples

  Selected classes (10):
   [ 0] apple pie
   [ 1] bibimbap
   [ 2] chicken wings
   [ 3] donuts
   [ 4] eggs benedict
   [ 5] french fries
   [ 6] grilled cheese sandwich
   [ 7] hamburger
   [ 8] ice cream
   [ 9] pizza


---
## 7. Load CLIP Model & Processor

In [9]:
print(f"Loading CLIP model '{CFG.model_id}'...")
PROCESSOR = AutoProcessor.from_pretrained(CFG.model_id)
MODEL     = CLIPModel.from_pretrained(CFG.model_id).to(DEVICE)
MODEL.eval()

n_params = sum(p.numel() for p in MODEL.parameters()) / 1e6
print(f"Model loaded — {n_params:.1f}M parameters.")

Loading CLIP model 'openai/clip-vit-base-patch32'...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21373.75it/s]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded — 151.3M parameters.


---
## 8. Feature Extraction

In [10]:
def create_image_dataloader(
    dataset: Dataset,
    batch_size: int,
    num_workers: int,
    shuffle: bool = False,
) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        collate_fn=image_collate_fn,
        pin_memory=torch.cuda.is_available(),
    )


def extract_image_features(
    model: CLIPModel,
    processor: AutoProcessor,
    dataloader: DataLoader,
    device: torch.device,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    feat_batches, lbl_batches, idx_batches = [], [], []
    model.eval()
    with torch.inference_mode():
        for batch in dataloader:
            inputs       = processor(images=batch["images"], return_tensors="pt")
            pixel_values = inputs["pixel_values"].to(device)
            outputs      = model.get_image_features(pixel_values=pixel_values)
            feats        = outputs.pooler_output if hasattr(outputs, "pooler_output") \
                           else (outputs[0] if isinstance(outputs, (list, tuple)) else outputs)
            feats = feats / feats.norm(dim=-1, keepdim=True)
            feat_batches.append(feats.cpu())
            lbl_batches.append(batch["labels"].cpu())
            idx_batches.append(batch["indices"].cpu())
    return (
        torch.cat(feat_batches, dim=0),
        torch.cat(lbl_batches,  dim=0),
        torch.cat(idx_batches,  dim=0),
    )


print("Feature extraction functions ready.")

Feature extraction functions ready.


In [11]:
print("Building dataloaders and extracting image features...")

dataloaders: Dict[str, DataLoader] = {
    split_name: create_image_dataloader(
        dataset=ds,
        batch_size=CFG.batch_size,
        num_workers=CFG.num_workers,
    )
    for split_name, ds in datasets_by_split.items()
}

split_features: Dict[str, torch.Tensor] = {}
split_labels:   Dict[str, torch.Tensor] = {}
split_indices:  Dict[str, torch.Tensor] = {}

for split_name, dataloader in dataloaders.items():
    print(f"  Extracting features for split '{split_name}'...")
    feats, labels, indices = extract_image_features(
        model=MODEL, processor=PROCESSOR,
        dataloader=dataloader, device=DEVICE,
    )
    split_features[split_name] = feats
    split_labels[split_name]   = labels
    split_indices[split_name]  = indices
    print(f"     shape: {feats.shape}")

print("\nFeature extraction complete.")

Building dataloaders and extracting image features...
  Extracting features for split 'few_shot_train'...
     shape: torch.Size([1280, 512])
  Extracting features for split 'few_shot_dev'...
     shape: torch.Size([200, 512])
  Extracting features for split 'validation'...


/Users/khoatran/Developer/learning-so-deep/venv/lib/python3.12/site-packages/PIL/TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


     shape: torch.Size([2500, 512])

Feature extraction complete.


---
## 9. Zero-Shot Classification

In [12]:
def build_zero_shot_text_features(
    model: CLIPModel,
    processor: AutoProcessor,
    class_names: Sequence[str],
    prompt_templates: Sequence[str],
    device: torch.device,
) -> torch.Tensor:
    text_feats: List[torch.Tensor] = []
    model.eval()
    with torch.inference_mode():
        for class_name in class_names:
            prompts      = [t.format(humanize_class_name(class_name)) for t in prompt_templates]
            text_inputs  = processor(text=prompts, return_tensors="pt",
                                     padding=True, truncation=True)
            text_inputs  = {k: v.to(device) for k, v in text_inputs.items()}
            text_outputs = model.get_text_features(**text_inputs)
            feats        = text_outputs.pooler_output if hasattr(text_outputs, "pooler_output") \
                           else (text_outputs[0] if isinstance(text_outputs, (list, tuple))
                                 else text_outputs)
            feats        = feats / feats.norm(dim=-1, keepdim=True)
            avg_feat     = feats.mean(dim=0)
            avg_feat     = avg_feat / avg_feat.norm()
            text_feats.append(avg_feat.cpu())
    return torch.stack(text_feats, dim=0)


print("Zero-shot text feature builder ready.")

Zero-shot text feature builder ready.


In [13]:
print("Building zero-shot text features...")
TEXT_FEATURES = build_zero_shot_text_features(
    model=MODEL,
    processor=PROCESSOR,
    class_names=CLASS_NAMES,
    prompt_templates=CFG.prompt_templates,
    device=DEVICE,
)
print(f"Text feature matrix shape: {TEXT_FEATURES.shape}")

torch.save(
    {"class_names": CLASS_NAMES,
     "prompt_templates": list(CFG.prompt_templates),
     "text_features": TEXT_FEATURES},
    OUTPUT_DIR / "zero_shot_text_features.pt",
)

Building zero-shot text features...
Text feature matrix shape: torch.Size([10, 512])


---
## 10. Evaluation Utilities

In [14]:
def compute_metrics(
    true_labels: Sequence[int],
    predicted_labels: Sequence[int],
    class_names: Sequence[str],
) -> Tuple[Dict[str, float], np.ndarray, Dict[str, Any]]:
    labels = list(range(len(class_names)))
    p_mac, r_mac, f1_mac, _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="macro",    zero_division=0)
    p_w,   r_w,   f1_w,   _ = precision_recall_fscore_support(
        true_labels, predicted_labels, average="weighted", zero_division=0)
    matrix = confusion_matrix(true_labels, predicted_labels, labels=labels)
    report = classification_report(
        true_labels, predicted_labels,
        labels=labels,
        target_names=[humanize_class_name(n) for n in class_names],
        zero_division=0, output_dict=True,
    )
    metrics = {
        "accuracy":           float(accuracy_score(true_labels, predicted_labels)),
        "balanced_accuracy":  float(balanced_accuracy_score(true_labels, predicted_labels)),
        "macro_precision":    float(p_mac),
        "macro_recall":       float(r_mac),
        "macro_f1":           float(f1_mac),
        "weighted_precision": float(p_w),
        "weighted_recall":    float(r_w),
        "weighted_f1":        float(f1_w),
    }
    return metrics, matrix, report


def evaluate_logits(
    logits: torch.Tensor,
    labels: torch.Tensor,
    indices: torch.Tensor,
    class_names: Sequence[str],
    method_name: str,
) -> Dict[str, Any]:
    probs       = torch.softmax(logits, dim=-1)
    conf, preds = probs.max(dim=-1)
    true_lbls   = labels.cpu().tolist()
    pred_lbls   = preds.cpu().tolist()
    sample_idxs = indices.cpu().tolist()

    metrics, matrix, report = compute_metrics(true_lbls, pred_lbls, class_names)

    pred_rows = [
        {
            "sample_index":       int(si),
            "method":             method_name,
            "true_label_id":      int(tl),
            "true_label_name":    class_names[tl],
            "predicted_label_id": int(pl),
            "predicted_label_name": class_names[pl],
            "confidence":         float(c),
        }
        for si, tl, pl, c in zip(sample_idxs, true_lbls, pred_lbls, conf.cpu().tolist())
    ]
    return {
        "metrics": metrics, "confusion_matrix": matrix,
        "classification_report": report, "predictions": pred_rows,
    }


def save_evaluation_artifacts(
    output_dir: Path,
    method_name: str,
    evaluation: Dict[str, Any],
    class_names: Sequence[str],
    show_plots: bool = True,
) -> None:
    method_dir = output_dir / method_name
    save_json(method_dir / "metrics.json", evaluation["metrics"])
    save_json(method_dir / "classification_report.json", evaluation["classification_report"])
    save_rows_csv(
        method_dir / "predictions.csv",
        evaluation["predictions"],
        fieldnames=["sample_index", "method", "true_label_id", "true_label_name",
                    "predicted_label_id", "predicted_label_name", "confidence"],
    )
    save_confusion_matrix_csv(
        method_dir / "confusion_matrix.csv",
        evaluation["confusion_matrix"],
        class_names,
    )
    save_confusion_matrix_plot(
        method_dir / "confusion_matrix.png",
        evaluation["confusion_matrix"],
        class_names,
        title=f"{method_name.replace('_', ' ').title()} Confusion Matrix",
        show=show_plots,
    )


print("Evaluation utilities defined.")

Evaluation utilities defined.


In [15]:
# ── Run zero-shot evaluation ───────────────────────────────────────────────
print("Evaluating zero-shot classifier on validation split...")
logit_scale = float(MODEL.logit_scale.exp().detach().cpu().item())
zero_shot_logits = logit_scale * (
    split_features["validation"] @ TEXT_FEATURES.T
)
zero_shot_eval = evaluate_logits(
    logits=zero_shot_logits,
    labels=split_labels["validation"],
    indices=split_indices["validation"],
    class_names=CLASS_NAMES,
    method_name="zero_shot",
)
save_evaluation_artifacts(OUTPUT_DIR, "zero_shot", zero_shot_eval, CLASS_NAMES)

print("\nZero-Shot Metrics:")
for k, v in zero_shot_eval["metrics"].items():
    print(f"   {k:<25} {v:.4f}")

Evaluating zero-shot classifier on validation split...

Zero-Shot Metrics:
   accuracy                  0.9796
   balanced_accuracy         0.9796
   macro_precision           0.9796
   macro_recall              0.9796
   macro_f1                  0.9796
   weighted_precision        0.9796
   weighted_recall           0.9796
   weighted_f1               0.9796


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_59378/1735540179.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 11. Few-Shot Linear Probe Training

In [16]:
def predict_linear_probe_logits(
    classifier: nn.Linear,
    features: torch.Tensor,
    batch_size: int,
    device: torch.device,
) -> torch.Tensor:
    logits_batches = []
    classifier.eval()
    with torch.inference_mode():
        for start in range(0, len(features), batch_size):
            batch_feats = features[start: start + batch_size].to(device)
            logits_batches.append(classifier(batch_feats).cpu())
    return torch.cat(logits_batches, dim=0)


def train_linear_probe(
    train_features: torch.Tensor,
    train_labels: torch.Tensor,
    dev_features: torch.Tensor,
    dev_labels: torch.Tensor,
    config: ExperimentConfig,
    class_names: Sequence[str],
    device: torch.device,
    output_dir: Path,
) -> Tuple[nn.Linear, List[Dict[str, Any]], Dict[str, Any]]:
    classifier = nn.Linear(train_features.shape[1], len(class_names)).to(device)
    optimizer  = torch.optim.AdamW(
        classifier.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
    )
    criterion   = nn.CrossEntropyLoss()
    train_bs    = min(config.batch_size * 4, len(train_features))
    train_loader = DataLoader(
        TensorDataset(train_features, train_labels),
        batch_size=train_bs, shuffle=True,
    )
    has_dev       = len(dev_features) > 0
    history_rows  = []
    best_state    = copy.deepcopy(classifier.state_dict())
    best_metrics  = {}
    best_macro_f1 = float("-inf")

    for epoch in range(1, config.epochs + 1):
        classifier.train()
        running_loss, seen = 0.0, 0

        for batch_feats, batch_lbls in train_loader:
            batch_feats = batch_feats.to(device)
            batch_lbls  = batch_lbls.to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = classifier(batch_feats)
            loss   = criterion(logits, batch_lbls)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_lbls.size(0)
            seen         += batch_lbls.size(0)

        record = {"epoch": epoch, "train_loss": running_loss / max(seen, 1)}

        if has_dev:
            dev_logits = predict_linear_probe_logits(
                classifier, dev_features, config.batch_size * 8, device
            )
            dev_loss = criterion(dev_logits, dev_labels).item()
            dev_eval = evaluate_logits(
                logits=dev_logits, labels=dev_labels,
                indices=torch.arange(len(dev_labels)),
                class_names=class_names, method_name="few_shot_dev",
            )
            record.update({
                "dev_loss":      dev_loss,
                "dev_accuracy":  dev_eval["metrics"]["accuracy"],
                "dev_macro_f1": dev_eval["metrics"]["macro_f1"],
            })
            if dev_eval["metrics"]["macro_f1"] > best_macro_f1:
                best_macro_f1 = dev_eval["metrics"]["macro_f1"]
                best_metrics  = dev_eval["metrics"]
                best_state    = {k: v.detach().cpu() for k, v in classifier.state_dict().items()}

            if epoch % 5 == 0 or epoch == config.epochs:
                print(
                    f"  Epoch {epoch:>3}/{config.epochs} "
                    f"| train_loss={record['train_loss']:.4f} "
                    f"| dev_loss={dev_loss:.4f} "
                    f"| dev_acc={record['dev_accuracy']:.4f} "
                    f"| dev_f1={record['dev_macro_f1']:.4f}"
                )
        else:
            best_state = {k: v.detach().cpu() for k, v in classifier.state_dict().items()}
            if epoch % 5 == 0 or epoch == config.epochs:
                print(f"  Epoch {epoch:>3}/{config.epochs} "
                      f"| train_loss={record['train_loss']:.4f}")

        history_rows.append(record)

    classifier.load_state_dict(best_state)
    torch.save(
        {
            "model_type":    "linear_probe",
            "feature_dim":   int(train_features.shape[1]),
            "num_classes":   len(class_names),
            "class_names":   list(class_names),
            "config":        config.to_serializable_dict(),
            "state_dict":    {k: v.detach().cpu() for k, v in classifier.state_dict().items()},
            "best_dev_metrics": best_metrics,
        },
        output_dir / "few_shot_linear_probe.pt",
    )
    return classifier, history_rows, best_metrics


print("Linear probe training functions defined.")

Linear probe training functions defined.


In [17]:
print("Training few-shot linear probe...\n")
CLASSIFIER, HISTORY, BEST_DEV_METRICS = train_linear_probe(
    train_features=split_features["few_shot_train"],
    train_labels=split_labels["few_shot_train"],
    dev_features=split_features["few_shot_dev"],
    dev_labels=split_labels["few_shot_dev"],
    config=CFG,
    class_names=CLASS_NAMES,
    device=DEVICE,
    output_dir=OUTPUT_DIR,
)

save_rows_csv(
    OUTPUT_DIR / "few_shot_training_history.csv",
    HISTORY,
    fieldnames=["epoch", "train_loss", "dev_loss", "dev_accuracy", "dev_macro_f1"],
)
print("\nTraining complete. Best dev metrics:")
for k, v in BEST_DEV_METRICS.items():
    print(f"   {k:<25} {v:.4f}")

Training few-shot linear probe...

  Epoch   5/20 | train_loss=1.9155 | dev_loss=1.8774 | dev_acc=0.9600 | dev_f1=0.9598
  Epoch  10/20 | train_loss=1.5442 | dev_loss=1.5170 | dev_acc=0.9650 | dev_f1=0.9648
  Epoch  15/20 | train_loss=1.2425 | dev_loss=1.2266 | dev_acc=0.9650 | dev_f1=0.9648
  Epoch  20/20 | train_loss=1.0084 | dev_loss=1.0012 | dev_acc=0.9650 | dev_f1=0.9650

Training complete. Best dev metrics:
   accuracy                  0.9700
   balanced_accuracy         0.9700
   macro_precision           0.9714
   macro_recall              0.9700
   macro_f1                  0.9698
   weighted_precision        0.9714
   weighted_recall           0.9700
   weighted_f1               0.9698


In [18]:
# ── Plot training curves ───────────────────────────────────────────────────
print("Plotting training history...")
plot_training_history(HISTORY, OUTPUT_DIR)

Plotting training history...
  → Training curves saved to output/clip-vit-base-patch32/20260326_170105/training_curves.png


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_59378/1735540179.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 12. Few-Shot Evaluation on Validation Set

In [19]:
print("Evaluating linear probe on validation split...")
few_shot_logits = predict_linear_probe_logits(
    classifier=CLASSIFIER,
    features=split_features["validation"],
    batch_size=CFG.batch_size * 8,
    device=DEVICE,
)
few_shot_eval = evaluate_logits(
    logits=few_shot_logits,
    labels=split_labels["validation"],
    indices=split_indices["validation"],
    class_names=CLASS_NAMES,
    method_name="few_shot_linear_probe",
)
save_evaluation_artifacts(
    OUTPUT_DIR, "few_shot_linear_probe", few_shot_eval, CLASS_NAMES
)

print("\nFew-Shot Linear Probe Metrics:")
for k, v in few_shot_eval["metrics"].items():
    print(f"   {k:<25} {v:.4f}")

Evaluating linear probe on validation split...

Few-Shot Linear Probe Metrics:
   accuracy                  0.9708
   balanced_accuracy         0.9708
   macro_precision           0.9710
   macro_recall              0.9708
   macro_f1                  0.9708
   weighted_precision        0.9710
   weighted_recall           0.9708
   weighted_f1               0.9708


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_59378/1735540179.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## 13. Method Comparison & Final Summary

In [20]:
# ── Save comparison CSV ────────────────────────────────────────────────────
comparison_rows = [
    {"method": "zero_shot",            **zero_shot_eval["metrics"]},
    {"method": "few_shot_linear_probe", **few_shot_eval["metrics"]},
]
save_rows_csv(
    OUTPUT_DIR / "comparison.csv",
    comparison_rows,
    fieldnames=[
        "method", "accuracy", "balanced_accuracy",
        "macro_precision", "macro_recall", "macro_f1",
        "weighted_precision", "weighted_recall", "weighted_f1",
    ],
)

# ── Deltas ─────────────────────────────────────────────────────────────────
metric_deltas = {
    k: few_shot_eval["metrics"][k] - zero_shot_eval["metrics"][k]
    for k in zero_shot_eval["metrics"]
}
comparison_summary = {
    "class_names":           CLASS_NAMES,
    "evaluation_split_name": "validation",
    "best_dev_metrics":       BEST_DEV_METRICS,
    "zero_shot":             zero_shot_eval["metrics"],
    "few_shot_linear_probe": few_shot_eval["metrics"],
    "metric_deltas":         metric_deltas,
}
save_json(OUTPUT_DIR / "comparison_summary.json", comparison_summary)

# ── Print table ────────────────────────────────────────────────────────────
print(f"{'Metric':<25} {'Zero-Shot':>10} {'Few-Shot':>10} {'Δ':>8}")
print("-" * 57)
for k in zero_shot_eval["metrics"]:
    zs = zero_shot_eval["metrics"][k]
    fs = few_shot_eval["metrics"][k]
    delta = fs - zs
    sign  = "+" if delta >= 0 else ""
    print(f"{k:<25} {zs:>10.4f} {fs:>10.4f} {sign}{delta:>7.4f}")

Metric                     Zero-Shot   Few-Shot        Δ
---------------------------------------------------------
accuracy                      0.9796     0.9708 -0.0088
balanced_accuracy             0.9796     0.9708 -0.0088
macro_precision               0.9796     0.9710 -0.0086
macro_recall                  0.9796     0.9708 -0.0088
macro_f1                      0.9796     0.9708 -0.0088
weighted_precision            0.9796     0.9710 -0.0086
weighted_recall               0.9796     0.9708 -0.0088
weighted_f1                   0.9796     0.9708 -0.0088


In [21]:
# ── Visual comparison bar chart ────────────────────────────────────────────
plot_comparison_bar(
    zero_shot_metrics=zero_shot_eval["metrics"],
    few_shot_metrics=few_shot_eval["metrics"],
    output_dir=OUTPUT_DIR,
)

  → Comparison bar chart saved to output/clip-vit-base-patch32/20260326_170105/comparison_bar.png


/var/folders/jr/_vnkd1dx5hz54ytwj94g23sr0000gn/T/ipykernel_59378/1735540179.py:133: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [22]:
# ── Final run summary ──────────────────────────────────────────────────────
final_summary = {
    "output_dir":         str(OUTPUT_DIR),
    "dataset_summary":    dataset_summary,
    "comparison_summary": comparison_summary,
}
save_json(OUTPUT_DIR / "run_summary.json", final_summary)

print("\n" + "=" * 60)
print("EXPERIMENT COMPLETE")
print("=" * 60)
print(f"  Output directory : {OUTPUT_DIR}")
print(f"  Zero-shot  acc   : {zero_shot_eval['metrics']['accuracy']:.4f}")
print(f"  Few-shot   acc   : {few_shot_eval['metrics']['accuracy']:.4f}")
print(f"  Δ accuracy       : {metric_deltas['accuracy']:+.4f}")
print("=" * 60)


EXPERIMENT COMPLETE
  Output directory : output/clip-vit-base-patch32/20260326_170105
  Zero-shot  acc   : 0.9796
  Few-shot   acc   : 0.9708
  Δ accuracy       : -0.0088


---
## 14. Output Artefact Map

```
outputs/food101_experiments/<TIMESTAMP>/
│
├── config.json                        ← full experiment config
├── dataset_summary.json               ← split sizes & class names
├── run_summary.json                   ← top-level result roll-up
├── comparison.csv                     ← metric table (both methods)
├── comparison_summary.json            ← metric deltas & best dev
├── few_shot_training_history.csv      ← per-epoch training log
├── few_shot_linear_probe.pt           ← model checkpoint
├── zero_shot_text_features.pt         ← text embeddings
├── training_curves.png                ← loss / metric curves
├── comparison_bar.png                 ← side-by-side bar chart
│
├── zero_shot/
│   ├── metrics.json
│   ├── classification_report.json
│   ├── predictions.csv
│   ├── confusion_matrix.csv
│   └── confusion_matrix.png
│
└── few_shot_linear_probe/
    ├── metrics.json
    ├── classification_report.json
    ├── predictions.csv
    ├── confusion_matrix.csv
    └── confusion_matrix.png
```

In [23]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent.resolve()   # .../multimodal
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from models import results as mm_results

run_name = f"report_{Path(OUTPUT_DIR).name}"
artifact_dir = mm_results.run_from_notebook(
    dataset=datasets_by_split,
    output_root=Path('output'),
    artifact_root=Path('../artifacts').resolve(),
    run_name=run_name,
    examples_per_method=2,
    model=MODEL,
    processor=PROCESSOR,
)
print(f"Saved consolidated artifacts to: {artifact_dir}")


Saved consolidated artifacts to: /Users/khoatran/Developer/learning-so-deep/assignments/assignment1/multimodal/artifacts/report_20260326_170105
